# 03 Semantic Perception + Geometric Feature Integration (11 September 2026)
Real nuScenes LiDAR frame -> processed N x 4 -> semantic source (lidarseg / annotation / model) -> project classes -> geometric features -> `PerceptionResult` -> `RegionFeatures` -> **unchanged** Importance Engine v1 -> Resolution Engine v1.

9/10 September work is reused unchanged (`lidar_loader`, `preprocessing`, `LiDARFrame`/`RegionFeatures` contracts, both engines). See `docs/semantic_perception_integration.md` for provenance and limitations.

> Honesty rules: annotation-derived labels are NOT ML predictions; non-ML sources carry NaN (not applicable) point-level confidence, never a fabricated score; no accuracy claim is made without prediction-vs-ground-truth evaluation.


## 1. Environment and project setup


In [7]:
import sys
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

print("Python:", sys.version)
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
# No heavy ML dependencies: the annotation/lidarseg path needs none.
# (A future model-backed run imports only what that model requires.)


Python: 3.14.4 (tags/v3.14.4:23116f9, Apr  7 2026, 14:10:54) [MSC v.1944 64 bit (AMD64)]
NumPy: 2.4.4
Pandas: 3.0.2


## 2. Project paths (Google Drive on Colab, checkout root locally)


In [8]:
try:
    from google.colab import drive
    drive.mount("/content/drive")
    PROJECT_ROOT = "/content/drive/MyDrive/LiDAR_Hackathon"
except ImportError:
    # Local execution: repository checkout root (parent of notebooks/).
    PROJECT_ROOT = os.path.abspath(os.path.join(os.getcwd(), ".."))
    print("Local PROJECT_ROOT:", PROJECT_ROOT)

SRC_ROOT = os.path.join(PROJECT_ROOT, "src")
RAW_ROOT = os.path.join(PROJECT_ROOT, "data", "raw")
PROCESSED_ROOT = os.path.join(PROJECT_ROOT, "data", "processed")
RESULTS_ROOT = os.path.join(PROJECT_ROOT, "results")
FIGURES_ROOT = os.path.join(RESULTS_ROOT, "figures")
PERCEPTION_DIR = os.path.join(RESULTS_ROOT, "perception_results")
# nuScenes dataroot: directory containing samples/, sweeps/, v1.0-mini/.
NUSC_ROOT = os.path.join(RAW_ROOT, "nuscenes")

os.makedirs(RESULTS_ROOT, exist_ok=True)
os.makedirs(FIGURES_ROOT, exist_ok=True)
os.makedirs(PERCEPTION_DIR, exist_ok=True)
print("SRC_ROOT:", SRC_ROOT)
print("PROCESSED_ROOT:", PROCESSED_ROOT)
print("NUSC_ROOT:", NUSC_ROOT)


Local PROJECT_ROOT: c:\Users\SARKAR\Documents\Default Project
SRC_ROOT: c:\Users\SARKAR\Documents\Default Project\src
PROCESSED_ROOT: c:\Users\SARKAR\Documents\Default Project\data\processed
NUSC_ROOT: c:\Users\SARKAR\Documents\Default Project\data\raw\nuscenes


## 3. Import existing project modules (reuse, never a second implementation)


In [9]:
import sys

if SRC_ROOT not in sys.path:
    sys.path.append(SRC_ROOT)
if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

from src.data_types import LiDARFrame, PerceptionResult, RegionFeatures
from src.interface_validator import validate_region_features, validate_perception_result
from src.importance_engine import ImportanceEngine
from src.resolution_engine import ResolutionEngine
# 11-September additive modules (new taxonomy + adapter only):
from src.semantic_mapping import (
    PROJECT_CLASSES, PROJECT_CLASS_NAMES, PROJECT_SEMANTIC_IMPORTANCE,
    PROJECT_DYNAMIC_PRIOR, VALID_SEMANTIC_SOURCES,
    map_nuscenes_label_to_project, map_lidarseg_id_to_project,
)
from src.semantic_adapter import (
    CELL_SIZE_M, annotations_to_sensor_frame, assign_annotation_semantics, build_perception_from_semantics,
    aggregate_to_regions, compute_point_geometry,
)

importance_engine = ImportanceEngine()  # default config, unchanged
resolution_engine = ResolutionEngine()  # default config, unchanged
print("Imports OK. Project classes:", sorted(PROJECT_CLASSES))



Imports OK. Project classes: ['pedestrian_vru', 'road_driveable', 'static_manmade', 'unknown', 'vegetation', 'vehicle']


## 4. Load one processed real nuScenes frame (10 September pipeline output)


In [10]:
# Ensure the processed-data directory exists and is the correct one.
PROCESSED_ROOT = globals().get(
    "PROCESSED_ROOT",
    os.path.join(PROJECT_ROOT, "data", "processed"),
)

processed_paths = []
seen_paths = set()

# Search the complete project tree because the processed files may be
# located in a nested directory rather than directly under data/processed.
for current_root, _, filenames in os.walk(PROJECT_ROOT):
    for filename in filenames:
        if filename.endswith("_LIDAR_TOP_xyzi.npy"):
            path = os.path.abspath(os.path.join(current_root, filename))
            if path not in seen_paths:
                processed_paths.append(path)
                seen_paths.add(path)

processed_paths.sort()

if not processed_paths:
    print("No processed LiDAR frames were found.")
    print("Expected filename pattern: *_LIDAR_TOP_xyzi.npy")
    print("Expected location:", os.path.join(PROJECT_ROOT, "data", "processed"))
    print("Run the preprocessing pipeline before executing this notebook.")
else:
    processed_path = processed_paths[0]
    PROCESSED_ROOT = os.path.dirname(processed_path)
    processed_files = [os.path.basename(path) for path in processed_paths]

    print("PROCESSED_ROOT:", PROCESSED_ROOT)
    print("Number of processed frames:", len(processed_files))
    print(processed_files[:10])

    processed_file = os.path.basename(processed_path)
    points = np.load(processed_path)
    print("Point shape:", points.shape)

    assert points.ndim == 2
    assert points.shape[1] == 4  # N x 4 [x, y, z, intensity]
    assert np.isfinite(points).all()


No processed LiDAR frames were found.
Expected filename pattern: *_LIDAR_TOP_xyzi.npy
Expected location: c:\Users\SARKAR\Documents\Default Project\data\processed
Run the preprocessing pipeline before executing this notebook.


## 5. Recover frame metadata


In [14]:
# Reconstruct sample_token from an available processed LiDAR file.
processed_file = None
processed_path = None

search_roots = [
    PROCESSED_ROOT,
    os.path.join(PROJECT_ROOT, "processed"),
    os.path.join(PROJECT_ROOT, "data", "processed"),
    os.path.join(PROJECT_ROOT, "notebooks", "data", "processed"),
    PROJECT_ROOT,
]

for search_root in search_roots:
    if not os.path.isdir(search_root):
        continue

    matches = []
    for root, _, files in os.walk(search_root):
        for name in files:
            if name.endswith("_LIDAR_TOP_xyzi.npy"):
                matches.append(os.path.join(root, name))

    if matches:
        processed_path = sorted(matches)[0]
        processed_file = os.path.basename(processed_path)
        PROCESSED_ROOT = os.path.dirname(processed_path)
        break

if processed_path is None:
    print("No processed LiDAR file was found.")
    print("Expected pattern: *_LIDAR_TOP_xyzi.npy")
    print("Expected directory:", os.path.join(PROJECT_ROOT, "data", "processed"))
    print("Run the preprocessing pipeline before continuing.")
    sample_token = None
    frame_id = None
else:
    sample_token = os.path.splitext(processed_file)[0].replace(
        "_LIDAR_TOP_xyzi", "")
    print("Using sample_token:", sample_token)
    print("Processed file:", processed_file)

    metadata_path = os.path.join(
        PROCESSED_ROOT, f"{sample_token}_metadata.csv")

    if not os.path.isfile(metadata_path):
        raise FileNotFoundError(
            f"Metadata file not found for {processed_file}: {metadata_path}")

    metadata_df = pd.read_csv(metadata_path)
    frame_metadata = metadata_df.iloc[0].to_dict()
    print("Frame metadata:")
    for key, value in frame_metadata.items():
        print(key, ":", value)

    frame_id = str(frame_metadata.get("frame_id", sample_token))


No processed LiDAR file was found.
Expected pattern: *_LIDAR_TOP_xyzi.npy
Expected directory: c:\Users\SARKAR\Documents\Default Project\data\processed
Run the preprocessing pipeline before continuing.


## 6. Load nuScenes metadata (verify it matches the processed frame)


In [15]:
# Colab: uncomment once if needed: !pip install -q nuscenes-devkit
from nuscenes.nuscenes import NuScenes

nusc = NuScenes(version="v1.0-mini", dataroot=NUSC_ROOT, verbose=False)
sample = nusc.get("sample", sample_token)
print("Sample token:", sample["token"])
print("Sample timestamp:", sample["timestamp"])
assert sample["token"] == sample_token, "Metadata/sample mismatch!"
print("Metadata corresponds to the processed frame.")


ModuleNotFoundError: No module named 'nuscenes'

## 7. Determine semantic-data availability (never assume lidarseg exists)


In [ ]:
# Robust lidarseg check: directory + actual label files + table (mini has none -> False, honest).
import glob as _glob
LIDARSEG_AVAILABLE = False
lidarseg_dir = os.path.join(NUSC_ROOT, 'lidarseg')
_lid_bins = _glob.glob(os.path.join(lidarseg_dir, '**', '*.bin'), recursive=True) if os.path.isdir(lidarseg_dir) else []
_lid_table = os.path.join(NUSC_ROOT, 'v1.0-mini', 'lidarseg.json')
if _lid_bins and os.path.isfile(_lid_table):
    print('Possible lidarseg directory:', lidarseg_dir, '| bins:', len(_lid_bins))
    LIDARSEG_AVAILABLE = True
else:
    print('lidarseg bins:', len(_lid_bins), '| lidarseg.json:', os.path.isfile(_lid_table))
# Resolve the LIDAR_TOP sample_data token (full schema sample[data] or mini sample_token match).
lidarseg_path = None
lidar_token_check = None
try:
    from src.semantic_adapter import _resolve_lidar_sd_token as _resolve_sd
    _sd_tok = sample['data']['LIDAR_TOP'] if isinstance(sample.get('data'), dict) and sample['data'].get('LIDAR_TOP') else _resolve_sd(nusc, sample)
    sample_data = nusc.get('sample_data', _sd_tok)
    lidar_token_check = sample_data['token']
    print('LIDAR_TOP sample_data token:', lidar_token_check)
    _cand = os.path.join(NUSC_ROOT, 'lidarseg', 'v1.0-mini', lidar_token_check + '_lidarseg.bin')
    if os.path.isfile(_cand):
        lidarseg_path = _cand
        LIDARSEG_AVAILABLE = True
        print('Per-sample lidarseg file:', _cand)
except Exception as exc:
    print('Could not resolve LIDAR_TOP sample_data:', exc)
print('LIDARSEG_AVAILABLE =', LIDARSEG_AVAILABLE)


## 8. Prepare project semantic class mapping (`src/semantic_mapping.py`)


In [ ]:
print("PROJECT_CLASSES:", PROJECT_CLASSES)
print("PROJECT_CLASS_NAMES:", PROJECT_CLASS_NAMES)
print("PROJECT_SEMANTIC_IMPORTANCE:", PROJECT_SEMANTIC_IMPORTANCE)
print("(Initial engineering importance parameters, not learned weights.)")
print("(Dynamic prior:", PROJECT_DYNAMIC_PRIOR, "- heuristic, not velocity.)")


## 9. nuScenes -> project class mapping (documented, no invented mappings)


In [ ]:
demo_labels = ["vehicle.car", "vehicle.truck", "human.pedestrian.adult",
               "movable_object.barrier", "static.manmade", "static.vegetation",
               "flat.driveable_surface", "noise", "flat.terrain"]
for lab in demo_labels:
    print(f"{lab:35s} -> {map_nuscenes_label_to_project(lab)}")


## 10. Load lidarseg if available (exact per-point alignment required)


In [ ]:
semantic_labels = None
semantic_source = None
original_labels = None

if LIDARSEG_AVAILABLE:
    # Resolve the lidarseg .bin for THIS sample_data record via devkit
    # conventions (label file parallels the point cloud; index i <-> point i).
    from nuscenes.utils.data_classes import LidarPointCloud  # noqa
    lidar_path, boxes, _ = nusc.get_sample_data(sample["data"]["LIDAR_TOP"])
    # lidarseg label file: <dataroot>/lidarseg/v1.0-mini/<lidar_token>_lidarseg.bin
    sd = nusc.get("sample_data", sample["data"]["LIDAR_TOP"])
    candidate = os.path.join(NUSC_ROOT, "lidarseg", "v1.0-mini",
                             f"{sd['token']}_lidarseg.bin")
    print("Lidarseg candidate:", candidate)
    lidarseg_ids = np.fromfile(candidate, dtype=np.uint8)
    print("lidarseg labels:", lidarseg_ids.shape)
    assert len(lidarseg_ids) == len(points), \
        "lidarseg/point count mismatch: wrong sample association!"
    proj, orig, ids = [], [], []
    for lid in lidarseg_ids.tolist():
        o, p = map_lidarseg_id_to_project(int(lid))
        orig.append(o); proj.append(p); ids.append(PROJECT_CLASSES[p])
    semantic_labels = np.array(proj, dtype=object)
    original_labels = np.array(orig, dtype=object)
    semantic_source = np.array(["lidarseg"] * len(points), dtype=object)
    print("Lidarseg branch: per-point labels aligned.")
else:
    print("Lidarseg unavailable -> annotation fallback (next section).")


## 11. Otherwise load object annotations (honest fallback, no faked segmentation)


In [ ]:
# Mini-safe annotation count (full schema sample[anns] or mini table match).
if isinstance(sample.get('anns'), list):
    ann_tokens = sample['anns']
else:
    ann_tokens = [r['token'] for r in (getattr(nusc, 'sample_annotation', []) or []) if isinstance(r, dict) and r.get('sample_token') == sample.get('token')]
print('Annotation count:', len(ann_tokens))
# Map boxes global -> LiDAR sensor frame (records are map-frame;
# points are sensor-frame -- direct comparison gives zero overlap).
annotations = annotations_to_sensor_frame(nusc, sample)
print('Example annotation (sensor frame):', annotations[0] if annotations else None)
semantic_labels, semantic_source, _ignored_conf = assign_annotation_semantics(points, annotations)
original_labels = np.array([''] * len(points), dtype=object)
# Points outside boxes stay unknown/fallback: never auto-labelled.
from collections import Counter as _C
print(_C(semantic_source.tolist()))
print(_C(semantic_labels.tolist()))


## 12. Source-aware semantic output + confidence rule (never fabricate confidence)


In [ ]:
assert semantic_labels is not None and semantic_source is not None
assert set(np.unique(semantic_source)).issubset(VALID_SEMANTIC_SOURCES)
print("Sources present:", sorted(set(semantic_source.tolist())))
# Rule: model -> real confidence; lidarseg/annotation/fallback -> NaN
# (not applicable). build_perception_from_semantics enforces this.
if "model" in set(semantic_source.tolist()):
    raise RuntimeError("Model-backed run must supply actual model confidence.")
print("Confidence for these sources: NaN (not applicable) -- enforced below.")


## 13. Build geometric features (distance, elevation)


In [ ]:
distance, elevation = compute_point_geometry(points)
print("distance: min/max/mean:", float(distance.min()), float(distance.max()),
      float(distance.mean()))
print("elevation: min/max/mean:", float(elevation.min()), float(elevation.max()),
      float(elevation.mean()))
# Region-level roughness/point_density are computed during aggregation.


## 14. Create PerceptionResult (existing project structure)


In [ ]:
perception_result, source_array, original_array = build_perception_from_semantics(
    frame_id=frame_id,
    points=points,
    semantic_labels=semantic_labels,
    semantic_source=semantic_source,
    confidence=None,  # non-ML sources: NaN enforced, nothing fabricated
    original_labels=original_labels,
)
print("PerceptionResult:", perception_result.points.shape,
      len(perception_result.semantic_labels))


## 15. Validate PerceptionResult


In [ ]:
assert perception_result.points.ndim == 2
assert perception_result.points.shape[1] == 4
assert len(perception_result.semantic_labels) == len(perception_result.points)
assert len(perception_result.confidence) == len(perception_result.points)
assert len(perception_result.distance) == len(perception_result.points)
assert len(perception_result.elevation) == len(perception_result.points)
assert np.isfinite(perception_result.points).all()
validate_perception_result(perception_result)
print("PerceptionResult valid.")


## 16. Aggregate point information into spatial regions (2.0 m grid, as 10 Sept)


In [ ]:
region_features, region_details = aggregate_to_regions(
    perception_result, source_array, cell_size=CELL_SIZE_M)
print("Number of regions:", len(region_features))
regions_df = pd.DataFrame(region_details)
print(regions_df[["region_id", "x", "y", "distance", "elevation",
                 "semantic_label", "semantic_source",
                 "dominant_class_ratio", "point_count"]].head().to_string(index=False))


## 17. Create RegionFeatures (done above) + interface validation


In [ ]:
for region in region_features:
    validate_region_features(region)
print("All RegionFeatures valid:", len(region_features))
print("NOTE: terrain_complexity (= roughness) is a geometric roughness-derived "
      "proxy, NOT a trained terrain classifier.")
print("NOTE: dynamic_relevance uses PROJECT_DYNAMIC_PRIOR heuristic unless "
      "genuine motion evidence exists.")


## 18. Connect RegionFeatures to Importance Engine + Resolution Engine (unchanged)


In [ ]:
importance_rows = []
for region in region_features:
    out = importance_engine.calculate(region)  # ImportanceResult object
    resolution_m = resolution_engine.assign_resolution(out.final_importance)
    importance_rows.append({
        "region_id": region.region_id, "x": region.x, "y": region.y,
        "distance": region.distance, "elevation": region.elevation,
        "terrain_complexity": region.roughness,
        "semantic_label": region.semantic_label,
        "semantic_importance": region.semantic_importance,
        "dynamic_relevance": region.dynamic_relevance,
        "uncertainty": region.uncertainty, "confidence": region.confidence,
        "point_count": region.point_count,
        "distance_score": out.distance_score,
        "base_importance": out.base_importance,
        "importance": out.final_importance, "resolution_m": resolution_m,
    })
importance_df = pd.DataFrame(importance_rows)
print(importance_df[["region_id", "semantic_label", "distance",
                    "importance", "resolution_m"]].head().to_string(index=False))


## 19. Verify semantic scenarios + comparison table (no fabrication)


In [ ]:
wanted = ["pedestrian_vru", "vehicle", "static_manmade", "road_driveable"]
for cls in wanted:
    sub = importance_df[importance_df["semantic_label"] == cls]
    if len(sub) == 0:
        print(f"{cls}: Class not observed in this frame.")
        continue
    row = sub.iloc[0]
    src = regions_df.loc[regions_df["region_id"] == row["region_id"],
                         "semantic_source"].iloc[0]
    print(f"{cls}: distance={row['distance']:.1f} "
          f"sem_imp={row['semantic_importance']:.2f} "
          f"terrain={row['terrain_complexity']:.3f} "
          f"dyn={row['dynamic_relevance']:.2f} unc={row['uncertainty']:.2f} "
          f"importance={row['importance']:.3f} res={row['resolution_m']} "
          f"source={src}")
rough = importance_df.sort_values("terrain_complexity", ascending=False).iloc[0]
print(f"Rough-terrain top: region={rough['region_id']} "
      f"terrain={rough['terrain_complexity']:.3f} "
      f"importance={rough['importance']:.3f} label={rough['semantic_label']}")
print("\nClass | Distance | SemImportance | Terrain | Importance | Resolution | Source")
for cls in wanted:
    sub = importance_df[importance_df["semantic_label"] == cls]
    if len(sub) == 0:
        print(f"{cls}: Class not observed in this frame.")
        continue
    r = sub.iloc[0]
    s = regions_df.loc[regions_df["region_id"] == r["region_id"],
                       "semantic_source"].iloc[0]
    print(f"{r['semantic_label']} | {r['distance']:.1f} | "
          f"{r['semantic_importance']:.2f} | {r['terrain_complexity']:.3f} | "
          f"{r['importance']:.3f} | {r['resolution_m']} | {s}")


## 20. Visualize semantic results, geometry, importance, resolution


In [ ]:
CLASS_COLORS = {"vehicle": "red", "pedestrian_vru": "magenta",
                "static_manmade": "orange", "vegetation": "green",
                "road_driveable": "gray", "unknown": "lightblue"}
xs = importance_df["x"].to_numpy(); ys = importance_df["y"].to_numpy()

# (a) semantic region map: where are the classes?
plt.figure(figsize=(7, 6))
for cls, color in CLASS_COLORS.items():
    sub = importance_df[importance_df["semantic_label"] == cls]
    plt.scatter(sub["x"], sub["y"], s=8, c=color, label=f"{cls} (n={len(sub)})")
plt.xlabel("x (m)"); plt.ylabel("y (m)")
plt.title(f"Semantic region map -- {frame_id}")
plt.legend(markerscale=3, fontsize=8); plt.axis("equal"); plt.tight_layout()
plt.savefig(os.path.join(FIGURES_ROOT, "semantic_region_map.png"), dpi=150)
plt.show()

# (b) elevation map
plt.figure(figsize=(7, 5))
plt.scatter(xs, ys, s=8, c=importance_df["elevation"].to_numpy(), cmap="viridis")
plt.colorbar(label="elevation (m)"); plt.xlabel("x (m)"); plt.ylabel("y (m)")
plt.title(f"Elevation map -- {frame_id}"); plt.axis("equal"); plt.tight_layout()
plt.savefig(os.path.join(FIGURES_ROOT, "elevation_map.png"), dpi=150)
plt.show()

# (c) terrain-complexity map
plt.figure(figsize=(7, 5))
plt.scatter(xs, ys, s=8, c=importance_df["terrain_complexity"].to_numpy(),
            cmap="copper", vmin=0, vmax=1)
plt.colorbar(label="terrain_complexity (roughness proxy)")
plt.xlabel("x (m)"); plt.ylabel("y (m)")
plt.title(f"Terrain-complexity map -- {frame_id}")
plt.axis("equal"); plt.tight_layout()
plt.savefig(os.path.join(FIGURES_ROOT, "terrain_complexity_map.png"), dpi=150)
plt.show()

# (d) importance: semantic + geometry -> importance
plt.figure(figsize=(7, 5))
plt.scatter(xs, ys, s=8, c=importance_df["importance"].to_numpy(),
            cmap="plasma", vmin=0, vmax=1)
plt.colorbar(label="final importance")
plt.xlabel("x (m)"); plt.ylabel("y (m)")
plt.title(f"Semantic+geometric importance -- {frame_id}")
plt.axis("equal"); plt.tight_layout()
plt.savefig(os.path.join(FIGURES_ROOT, "semantic_geometric_importance.png"), dpi=150)
plt.show()

# (e) resolution: importance -> resolution
plt.figure(figsize=(7, 5))
plt.scatter(xs, ys, s=8, c=importance_df["resolution_m"].to_numpy(),
            cmap="cool", vmin=0, vmax=0.5)
plt.colorbar(label="resolution (m)")
plt.xlabel("x (m)"); plt.ylabel("y (m)")
plt.title(f"Semantic+geometric resolution -- {frame_id}")
plt.axis("equal"); plt.tight_layout()
plt.savefig(os.path.join(FIGURES_ROOT, "semantic_geometric_resolution.png"), dpi=150)
plt.show()
print("Figures saved to", FIGURES_ROOT)


## 21. Test multiple real frames (same pipeline, no frame-specific code)


In [ ]:
def process_frame(points_in, frame_id_in, labels_in, sources_in):
    pr, src_arr, _ = build_perception_from_semantics(
        frame_id_in, points_in, labels_in, sources_in)
    regs, dets = aggregate_to_regions(pr, src_arr, cell_size=CELL_SIZE_M)
    for r in regs:
        validate_region_features(r)
    rows = []
    for r in regs:
        o = importance_engine.calculate(r)
        rows.append({"importance": o.final_importance,
                     "resolution_m": resolution_engine.assign_resolution(
                         o.final_importance),
                     "semantic_label": r.semantic_label})
    df = pd.DataFrame(rows)
    return {"frame_id": frame_id_in, "point_count": len(points_in),
            "region_count": len(regs),
            "semantic_source": ",".join(sorted(set(src_arr.tolist()))),
            "semantic_regions": int((df['semantic_label'] != 'unknown').sum()),
            "mean_importance": float(df["importance"].mean()),
            "min_importance": float(df["importance"].min()),
            "max_importance": float(df["importance"].max()),
            "resolution_dist": df["resolution_m"].value_counts().to_dict()}

frame_summaries = [process_frame(points, frame_id, semantic_labels, semantic_source)]
# Reuse the identical pipeline on up to 2 more real frames when available.
for extra_file in processed_files[1:3]:
    tok = extra_file.replace("_LIDAR_TOP_xyzi.npy", "")
    pts = np.load(os.path.join(PROCESSED_ROOT, extra_file))
    assert pts.ndim == 2 and pts.shape[1] == 4 and np.isfinite(pts).all()
    ex_sample = nusc.get("sample", tok)
    if LIDARSEG_AVAILABLE:
        raise RuntimeError("Multi-frame lidarseg path: resolve per-frame "
                           "label file as in Section 10 (not bulk-copied).")
    ex_anns = annotations_to_sensor_frame(nusc, ex_sample)
    ex_labels, ex_sources, _ = assign_annotation_semantics(pts, ex_anns)
    frame_summaries.append(process_frame(pts, tok, ex_labels, ex_sources))
multi_df = pd.DataFrame(frame_summaries)
print(multi_df.to_string(index=False))



## 22. Save results + semantic-source audit


In [ ]:
regions_df.to_csv(os.path.join(RESULTS_ROOT, "semantic_region_results.csv"), index=False)
importance_df.to_csv(os.path.join(RESULTS_ROOT, "real_importance_results.csv"), index=False)
np.save(os.path.join(PERCEPTION_DIR, f"{frame_id}_semantic_labels.npy"),
        np.asarray(semantic_labels, dtype=object))
np.save(os.path.join(PERCEPTION_DIR, f"{frame_id}_semantic_source.npy"),
        np.asarray(semantic_source, dtype=object))
source_counts = pd.Series(np.asarray(semantic_source, dtype=object)).value_counts()
print(source_counts)
source_counts.to_csv(os.path.join(RESULTS_ROOT, "semantic_source_audit.csv"),
                     header=["count"])
print("Saved: semantic_region_results.csv, real_importance_results.csv, "
      "perception_results/, semantic_source_audit.csv, figures/")


## 23. Document limitations
Full record: `docs/semantic_perception_integration.md`.

- Dataset: nuScenes v1.0-mini.
- Semantic source actually used is recorded per point/region (`lidarseg` / `annotation` / `model` / `fallback`).
- Annotation-derived labels are NOT ML predictions and carry NO calibrated ML confidence (NaN = not applicable).
- No semantic-segmentation accuracy claim without prediction-vs-ground-truth evaluation.
- Dynamic relevance may be a heuristic prior when motion is unavailable.
- Terrain complexity is a geometric roughness proxy, not a classifier.
- Semantic-importance values and engine weights remain prototype parameters.


In [ ]:
print("Limitations recorded (see docs/semantic_perception_integration.md).")
print("Semantic source used:", sorted(set(np.asarray(semantic_source, dtype=object).tolist())))
print("LIDARSEG_AVAILABLE =", LIDARSEG_AVAILABLE)


## 24. Final automated validation


In [ ]:
assert points.ndim == 2
assert points.shape[1] == 4
assert np.isfinite(points).all()
assert len(region_features) > 0
assert len(perception_result.semantic_labels) == len(points)
assert len(perception_result.distance) == len(points)
assert len(perception_result.elevation) == len(points)
for region in region_features:
    validate_region_features(region)
assert importance_df["importance"].between(0.0, 1.0).all()
assert importance_df["resolution_m"].isin([0.05, 0.10, 0.20, 0.50]).all()
print("11 SEPTEMBER SEMANTIC INTEGRATION: PASS")
